# Semana 14 — Interpretabilidad, ética e implicaciones

**Curso:** Ciencias de los Datos para las Finanzas, la Economía y los Negocios  
**Proyecto:** Propensión al consumo de marihuana en Colombia.  
**Modelo final:** Gradient Boosting Machine tuneado (semanas 10–12).

Este notebook implementa el entregable de la Semana 14:

1. **Interpretabilidad técnica** — valores SHAP sobre el GBM tuneado.
2. **Análisis de robustez** — reentrenamiento con una definición alternativa estricta de la variable objetivo $Y_{\text{alt}}=\mathbb{1}\{K\_03=1\}$.
3. **Ética, sesgos y limitaciones** — discusión estructurada sobre uso responsable.
4. **Implicaciones económicas y de política pública**.

El bloque de cálculo está encapsulado en `cannabis_tax.analysis.interpretability` para que el notebook sea reproducible y conciso.

## 0. Librerías e importaciones

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display, Markdown

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from cannabis_tax.analysis import interpretability as itp

FIG = project_root / 'reports' / 'figures'
TAB = project_root / 'reports' / 'tables'
summary_path = project_root / 'reports' / 'interpretability_summary.json'

## 1. Carga del modelo final y datos

Se reconstruye el GBM tuneado con los hiperparámetros encontrados en la Semana 12 (`RandomizedSearchCV`, 60 iteraciones × 5 folds).  
Se usan exactamente las mismas variables del benchmark (Entregable 6) para una comparación limpia:

$$\{\text{edad},\ \text{edad}^2,\ \text{sexo},\ \text{educación}\}.$$

In [ ]:
df = itp.load_data()
df = itp.attach_k03(df)
X, y, feat = itp._prepare(df, itp.TARGET)
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
gbm = itp.fit_gbm(X_tr, y_tr)
print('Hiperparámetros del GBM tuneado:')
for k, v in itp.GBM_TUNED_PARAMS.items():
    print(f'  {k:20s}: {v}')
print(f'\nMuestra modelada: train={len(X_tr)}, test={len(X_te)}')
print(f'Prevalencia (Y=1): {y.mean():.3f}')

## 2. Interpretabilidad con SHAP

Los valores SHAP (Lundberg & Lee, 2017) reparten la predicción del modelo entre las variables explicativas con una propiedad clave: la suma de los SHAP de un individuo más el valor base reproduce exactamente la predicción del modelo. Esto permite leer, para cada predicción, **qué variable empuja la propensión hacia arriba o hacia abajo y en qué magnitud**.

Ejecutamos el análisis sobre el conjunto de prueba para evitar contaminar la importancia con observaciones del entrenamiento.

In [ ]:
shap_out = itp.shap_analysis(gbm, X_tr, X_te, feat)
shap_out['importance']

### 2.1 Importancia global (SHAP summary)

In [ ]:
display(Image(filename=str(FIG / 'shap_summary.png')))

In [ ]:
display(Image(filename=str(FIG / 'shap_bar.png')))

**Lectura económica.** La edad y su término cuadrático concentran cerca del **77 %** de la importancia absoluta. Esto es coherente con la literatura: la propensión observada al consumo no es monótona en la edad sino concentrada en cohortes jóvenes y descendente con la edad, un patrón que el benchmark lineal captura sólo parcialmente (de ahí la ganancia del GBM tuneado al permitir no-linealidades). El sexo aporta 15 % adicional (la categoría *Mujer* empuja sistemáticamente la propensión hacia abajo), y la educación contribuye marginalmente (≈ 8 % entre las dos dummies).

Estas magnitudes alertan sobre un riesgo: con un *feature set* tan delgado, **el modelo segrega individuos casi exclusivamente por edad y sexo**. Cualquier uso operacional debe leerse a la luz de esa limitación.

### 2.2 Dependencia parcial — edad

In [ ]:
display(Image(filename=str(FIG / 'shap_dependence_edad.png')))

El gráfico de dependencia confirma la forma funcional implícita: la contribución al log-odds es positiva en cohortes jóvenes y decae conforme la edad aumenta. El benchmark MCO/Probit aproxima esta relación con $\beta_1 \cdot \text{edad} + \beta_2 \cdot \text{edad}^2$, pero el GBM la captura por tramos, lo cual explica que tras el tuning el GBM supere al benchmark en AUC-ROC y F1.

## 3. Análisis de robustez — definición alternativa de la variable objetivo

El `AUDIT_REPORT.md` (24 de marzo) documentó una discrepancia entre la variable `consumo_12m` usada en el modelo y la pregunta original `K_03` ("¿Ha consumido marihuana en los últimos 12 meses?") del cuestionario crudo. La distribución es:

| Fuente | Sí | No | NS/NR |
|---|---:|---:|---:|
| `K_03` (raw) | 1 223 | 2 754 | 5 |
| `consumo_12m` (base limpia) | 1 719 | 2 263 (NaN tratados como 0) | — |

El cruce muestra que **898 personas marcadas como consumidoras en `consumo_12m` declararon `K_03=2` (No)**. Esto sugiere que la base limpia construyó `consumo_12m` con una lógica más amplia (probablemente combinando señales de `consumo_vida`, `consumo_30d` y `consumo_reciente`).

Por integridad metodológica reentrenamos el GBM tuneado con $Y_{\text{alt}}=\mathbb{1}\{K\_03=1\}$ y comparamos.

In [ ]:
summary = json.loads(summary_path.read_text())
comp = pd.DataFrame({
    'Y original (consumo_12m)': summary['y_original'],
    'Y estricta (K_03=1)':       summary['y_strict_k03'],
}).round(4)
comp['Δ'] = (comp['Y estricta (K_03=1)'] - comp['Y original (consumo_12m)']).round(4)
comp

**Lectura.** Las dos especificaciones cuentan historias predictivas consistentes:

- La **CV AUC-10fold** mejora ligeramente con la Y estricta (0.6951 vs 0.6685, +0.027). Es decir, cuando la Y es operacionalmente más limpia, la generalización del modelo aumenta.
- La **AUC-ROC en test** cae ligeramente (−0.031), pero permanece por encima del azar y dentro del rango de las observaciones individuales de CV.
- El **F1 cae fuerte** (−0.29) porque con Y estricta la prevalencia baja de 0.43 a 0.31; el umbral de 0.5 deja de ser óptimo y muchos positivos se pierden. La caída no refleja peor discriminación (AUC sigue alto) sino mayor desbalance.

**Conclusión de robustez.** Las relaciones estructurales detectadas por el modelo (edad, sexo y educación como predictores de la propensión) **no dependen de la elección entre `consumo_12m` y la definición estricta `K_03=1`**. La selección final del GBM tuneado se mantiene; la discrepancia documental requiere ser declarada con honestidad en el paper como limitación.

## 4. Ética, sesgos y uso responsable

### 4.1 Sesgos de los datos

**Autoreporte y deseabilidad social.** La encuesta nacional sobre sustancias psicoactivas captura comportamientos estigmatizados. Es bien documentado que el autoreporte de consumo subestima la prevalencia real, y que ese subreporte está correlacionado con la percepción de riesgo legal y social: zonas con mayor presencia institucional o entornos familiares más conservadores tienden a subreportar más. Por construcción, el modelo aprende la propensión **a reportar consumo**, no la propensión a **consumir**.

**Sesgo de selección por la submuestra con precio.** En la fase de benchmark observamos que la muestra con precio (~755 obs) pierde representatividad y altera los coeficientes de educación. El GBM tuneado evita este problema trabajando sobre la muestra completa (~3 980 obs), pero a costa de no poder incorporar precio. Cualquier inferencia con elasticidad-precio queda fuera de alcance.

**Sesgo de medición en la variable objetivo.** La discrepancia documentada en la §3 entre `consumo_12m` y `K_03` introduce ruido de medición en la Y; la robustez muestra que ese ruido no domina el resultado, pero un trabajo definitivo debería reconciliar la construcción de la variable.

### 4.2 Riesgos del uso del modelo

1. **Perfilado individual.** El modelo predice una probabilidad de consumo a partir de variables demográficas. Usarlo para decisiones sobre personas (perfilado policial, decisiones laborales, vigilancia escolar) sería un mal uso: la incertidumbre individual es alta (AUC ≈ 0.73 implica un solapamiento sustancial entre distribuciones), y el costo social del falso positivo es severo dado el estigma asociado.
2. **Asociaciones estadísticas, no causales.** Los SHAP muestran *asociaciones* entre características y probabilidad estimada. Que el modelo asigne mayor probabilidad a hombres jóvenes no significa que el sexo o la edad *causen* el consumo; ambos están correlacionados con factores no observados (red social, exposición, entorno).
3. **Estigma reforzado.** Publicar mapas o perfiles de consumo puede reforzar estereotipos. Cualquier comunicación pública del modelo debería privilegiar lecturas agregadas (escenarios, elasticidades estimadas con cautela) sobre lecturas individuales.

### 4.3 Uso responsable propuesto

El modelo se propone como herramienta **agregada** para:

- Estimar tamaños de mercado bajo escenarios regulatorios alternativos.
- Apoyar el diseño tributario simulando bases de contribuyentes por estratos demográficos amplios.
- Identificar segmentos donde la política pública (preventiva o de regulación) podría priorizarse.

No se recomienda su uso para **decisiones individuales** o para focalización geográfica fina sin un análisis adicional de equidad y de capacidad institucional para gestionar falsos positivos.

## 5. Implicaciones económicas y de política pública

**Tamaño y demografía del mercado potencial.** Con un AUC-ROC ≈ 0.73, el GBM tuneado discrimina razonablemente individuos con alta vs baja propensión a reportar consumo. Esto permite *clasificar la población encuestada en grupos de mayor o menor propensión*, lo cual es insumo para una proyección poblacional con datos del DANE (composición etaria, sexo y educación a nivel nacional).

**Diseño tributario.** Si Colombia legalizara la distribución, la base potencial de contribuyentes (vía IVA u otro impuesto al consumo) estaría concentrada en cohortes jóvenes-masculinas, lo cual sugiere una elasticidad-precio probablemente elevada en ese segmento. Un impuesto demasiado alto desplazaría la demanda al mercado ilegal; uno demasiado bajo dejaría dinero sobre la mesa. La proyección de recaudo requiere acoplar este modelo de propensión con una elasticidad-precio creíble (que la submuestra disponible no permite estimar con limpieza).

**Política preventiva.** Las cohortes jóvenes (≈ 18–25 años) son las de mayor propensión observada y también las más sensibles a intervenciones tempranas. Esto justifica focalizar campañas de información y prevención en ese rango.

**Limitaciones para la política.** La ausencia de variables de entorno (geografía, red social, ingreso del hogar) limita la utilidad del modelo para diseñar intervenciones territoriales finas. Esta es la principal línea de trabajo futuro.

## 6. Resumen ejecutivo

| Dimensión | Hallazgo |
|---|---|
| Modelo final | GBM tuneado, AUC-ROC 0.7278 (test), supera a MCO/Probit en AUC y F1 |
| Determinante dominante | Edad y edad² concentran ≈ 77 % de la importancia SHAP |
| Robustez Y alternativa | CV AUC mejora (+0.027) con Y estricta K_03=1 — relaciones estables |
| Discrepancia documental | 898 individuos con `consumo_12m=1` pero `K_03=2` — declarada como limitación |
| Riesgo ético principal | Perfilado individual y refuerzo de estigma |
| Uso recomendado | Análisis agregado para diseño tributario y política preventiva |

Las tablas LaTeX (`reports/tables/shap_importance.tex`, `reports/tables/robustez_y_alt.tex`) y las figuras en `reports/figures/` quedan listas para integrarse al informe final.